In [ ]:
# Example 1: Geometry / Geography / Raster type construction via the public API.
from geoalchemy2 import Geometry, Geography, Raster

geom = Geometry(geometry_type='POINT', srid=4326)
geog = Geography(geometry_type='POLYGON', srid=4326)
rast = Raster()

print(f"Geometry: type={geom.geometry_type} srid={geom.srid}")
print(f"Geography: type={geog.geometry_type} srid={geog.srid}")
print(f"Raster: {type(rast).__name__}")

assert geom.geometry_type == 'POINT'
assert geom.srid == 4326
assert geog.geometry_type == 'POLYGON'

In [ ]:
# Example 2: WKTElement and WKBElement (spatial element wrappers).
from geoalchemy2 import WKTElement, WKBElement

wkt = WKTElement('POINT(1 2)', srid=4326)
print(f"WKTElement data: {wkt.data}")
print(f"WKTElement srid: {wkt.srid}")
print(f"WKTElement desc: {wkt.desc}")

# Well-known-binary hex for POINT(1 2).
wkb_hex = '0101000000000000000000F03F0000000000000040'
wkb = WKBElement(wkb_hex, srid=4326)
print(f"WKBElement srid: {wkb.srid}")
print(f"WKBElement desc length: {len(wkb.desc)}")

assert wkt.data == 'POINT(1 2)'
assert wkt.srid == 4326
assert wkb.srid == 4326

In [ ]:
# Example 3: Define an ORM model with a Geometry column.
from sqlalchemy.orm import declarative_base
from sqlalchemy import Column, Integer, String
from geoalchemy2 import Geometry

Base = declarative_base()

class Landmark(Base):
    __tablename__ = 'landmark'
    id = Column(Integer, primary_key=True)
    name = Column(String)
    geom = Column(Geometry(geometry_type='POINT', srid=4326))

cols = {c.name: type(c.type).__name__ for c in Landmark.__table__.columns}
print(f"Landmark columns: {cols}")

assert 'geom' in cols
assert cols['geom'] == 'Geometry'
assert Landmark.__table__.columns['geom'].type.srid == 4326

In [ ]:
# Example 4: Build a spatial SQL expression with geoalchemy2.functions.
from sqlalchemy import select, Column, Integer
from sqlalchemy.orm import declarative_base
from geoalchemy2 import Geometry, WKTElement
from geoalchemy2 import functions as gfunc

Base = declarative_base()

class Place(Base):
    __tablename__ = 'place'
    id = Column(Integer, primary_key=True)
    geom = Column(Geometry('POINT', srid=4326))

point = WKTElement('POINT(0 0)', srid=4326)
stmt = select(Place).where(gfunc.ST_DWithin(Place.geom, point, 1000))
compiled = str(stmt.compile(compile_kwargs={'literal_binds': False}))
print(compiled)

assert 'ST_DWithin' in compiled
assert 'place' in compiled.lower()

In [ ]:
# Example 5: Compile a SELECT involving multiple ST_* spatial functions.
from sqlalchemy import select, Column, Integer, String
from sqlalchemy.orm import declarative_base
from geoalchemy2 import Geometry, WKTElement
from geoalchemy2 import functions as gfunc

Base = declarative_base()

class Parcel(Base):
    __tablename__ = 'parcel'
    id = Column(Integer, primary_key=True)
    label = Column(String)
    shape = Column(Geometry('POLYGON', srid=4326))

probe = WKTElement('POINT(10 20)', srid=4326)
stmt = (
    select(Parcel.id, gfunc.ST_Area(Parcel.shape).label('area'))
    .where(gfunc.ST_Contains(Parcel.shape, probe))
)
compiled = str(stmt.compile())
print(compiled)

assert 'ST_Area' in compiled
assert 'ST_Contains' in compiled
assert 'parcel' in compiled.lower()